In [72]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import dagshub
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import SMOTENC, SMOTE
from imblearn.under_sampling import TomekLinks

from dotenv import load_dotenv
load_dotenv('../.env', override=True)

import os
DAGSHUB_USERNAME  = os.environ['DAGSHUB_USERNAME']
DAGSHUB_REPO_NAME = os.environ['DAGSHUB_REPO_NAME']
DAGSHUB_TOKEN     = os.environ['DAGSHUB_TOKEN']
MLFLOW_EXPERIMENT = os.getenv('MLFLOW_EXPERIMENT', 'wine-quality')
REGISTERED_MODEL  = os.getenv('MLFLOW_MODEL_NAME',  'wine-quality')

print("✅ Imports OK")


✅ Imports OK


## Carregando Dataset

In [73]:
df = pd.read_csv("../data/wine_quality_merged.csv")
# df = pd.read_csv("../data/winequality-red.csv")
print('Classes :', sorted(df['quality'].unique()))
print(df['quality'].value_counts().sort_index().to_string())   


Classes : [np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
quality
3      30
4     216
5    2138
6    2836
7    1079
8     193
9       5


## 2. Fusão de Classes
| Label | Classe original | Descrição |
|-------|-----------------|-----------|
| **0** | 3, 4, 5 | Ruim |
| **1** | 6 | Médio |
| **2** | 7, 8, 9 | Bom |


In [74]:
df_merged = df.copy()
df_merged["quality"] = df["quality"].apply(lambda x: "0" if x <= 5 else ("1" if x == 6 else "2"))
df_merged['quality'].replace("0",0, inplace=True)
df_merged['quality'].replace("1",1, inplace=True)
df_merged['quality'].replace("2",2, inplace=True)

# separar X e y
X = df_merged.drop(columns=['quality'])
y_orig = df_merged['quality']
print('Classes após fusão:', sorted(df_merged['quality'].unique()))
print(df_merged['quality'].value_counts().sort_index().to_string())



Classes após fusão: [np.int64(0), np.int64(1), np.int64(2)]
quality
0    2384
1    2836
2    1277


## 3. Split Treino / Validação / Teste  (60 / 20 / 20)

In [75]:
# 1ª divisão: 80% temp + 20% teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_orig, test_size=0.20, random_state=42, stratify=y_orig
)
# 2ª divisão: 75% treino + 25% validação  →  60% / 20% do total
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

total = len(X)
print(f"Treino:    {len(X_train):>5} amostras  ({len(X_train)/total*100:.1f}%)")
print(f"Validação: {len(X_val):>5} amostras  ({len(X_val)/total*100:.1f}%)")
print(f"Teste:     {len(X_test):>5} amostras  ({len(X_test)/total*100:.1f}%)")
print("\nDistribuição no treino:")
print(y_train.value_counts().sort_index())


Treino:     3897 amostras  (60.0%)
Validação:  1300 amostras  (20.0%)
Teste:      1300 amostras  (20.0%)

Distribuição no treino:
quality
0    1430
1    1701
2     766
Name: count, dtype: int64


### Models

In [76]:
# ── Identificar colunas ──────────────────────────────────────
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
cat_idx  = [X.columns.get_loc(c) for c in cat_cols]

print(f"Numéricas  ({len(num_cols)}): {num_cols}")
print(f"Categóricas({len(cat_cols)}): {cat_cols}")

# ── Fábrica de pré-processador ───────────────────────────────
def make_preprocessor():
    steps = [("num", StandardScaler(), num_cols)]
    if cat_cols:
        steps.append(("cat",
                       OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                       cat_cols))
    return ColumnTransformer(steps)

# ── Fábrica de modelos (instâncias frescas) ───────────────────
def get_models():
    return {
        "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
        "random_forest":       RandomForestClassifier(n_estimators=100, random_state=42),
        "knn":                 KNeighborsClassifier(),
        "svm":                 SVC(random_state=42),
        "xgboost":             XGBClassifier(eval_metric="mlogloss", random_state=42),
        "mlp":                 MLPClassifier(hidden_layer_sizes=(200, 100),
                                             max_iter=2000, random_state=42),
    }

# ── Cálculo de métricas ───────────────────────────────────────
def compute_metrics(y_true, y_pred, prefix=""):
    return {
        f"{prefix}accuracy":  round(accuracy_score(y_true, y_pred), 4),
        f"{prefix}precision": round(precision_score(y_true, y_pred,
                                                    average="weighted",
                                                    zero_division=0), 4),
        f"{prefix}recall":    round(recall_score(y_true, y_pred,
                                                 average="weighted",
                                                 zero_division=0), 4),
        f"{prefix}f1":        round(f1_score(y_true, y_pred,
                                             average="weighted",
                                             zero_division=0), 4),
    }

print("✅ Utilitários definidos")


Numéricas  (11): ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
Categóricas(1): ['type']
✅ Utilitários definidos


## 5. Configuração DagHub / MLflow

In [77]:
dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
mlflow.set_experiment("wine_quality_classification")

print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())


Initialized MLflow to track repo "frpbotero/wine-quality"

Repository frpbotero/wine-quality initialized!

✅ MLflow tracking URI: https://dagshub.com/frpbotero/wine-quality.mlflow


---
## Estratégia 1 — SMOTE (Over-sampling)
Aplica **SMOTENC** (suporte a features categóricas) no conjunto de treino para  
balancear as classes.  
Cada modelo é embrulhado num `Pipeline` com pré-processador próprio.


### 6. Aplicar SMOTE ao conjunto de treino

In [78]:
if cat_idx:
    sampler = SMOTENC(categorical_features=cat_idx, random_state=42)
    print("Usando SMOTENC (há features categóricas)")
else:
    sampler = SMOTE(random_state=42)
    print("Usando SMOTE (apenas features numéricas)")

X_train_smote, y_train_smote = sampler.fit_resample(X_train, y_train)

# Manter como DataFrame para compatibilidade com o Pipeline
X_train_smote = pd.DataFrame(X_train_smote, columns=X_train.columns)
y_train_smote = np.array(y_train_smote, dtype=int)

print("\nDistribuição após SMOTE:")
print(pd.Series(y_train_smote).value_counts().sort_index())
print(f"\nTotal de amostras: {len(X_train_smote)}")


Usando SMOTENC (há features categóricas)

Distribuição após SMOTE:
0    1701
1    1701
2    1701
Name: count, dtype: int64

Total de amostras: 5103


### 7. Treinar, validar, testar e registrar — SMOTE

In [79]:
STRATEGY = "SMOTE"
results_smote = {}

for model_name, model in get_models().items():
    print(f"\n{'='*60}")
    print(f"  Estratégia: {STRATEGY}  |  Modelo: {model_name}")
    print(f"{'='*60}")

    # ── Pipeline: pré-processador + classificador ─────────────
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("classifier",   model),
    ])

    # ── Treino ────────────────────────────────────────────────
    pipeline.fit(X_train_smote, y_train_smote)

    # ── Validação ─────────────────────────────────────────────
    y_val_pred  = pipeline.predict(X_val)
    val_metrics = compute_metrics(y_val, y_val_pred, prefix="val_")

    # ── Teste ─────────────────────────────────────────────────
    y_test_pred  = pipeline.predict(X_test)
    test_metrics = compute_metrics(y_test, y_test_pred, prefix="test_")

    all_metrics = {**val_metrics, **test_metrics}
    results_smote[model_name] = all_metrics

    # ── Imprimir resultados ───────────────────────────────────
    print(f"  Validação → Acc:{val_metrics['val_accuracy']:.4f}  "
          f"P:{val_metrics['val_precision']:.4f}  "
          f"R:{val_metrics['val_recall']:.4f}  "
          f"F1:{val_metrics['val_f1']:.4f}")
    print(f"  Teste     → Acc:{test_metrics['test_accuracy']:.4f}  "
          f"P:{test_metrics['test_precision']:.4f}  "
          f"R:{test_metrics['test_recall']:.4f}  "
          f"F1:{test_metrics['test_f1']:.4f}")
    print("\n" + classification_report(y_test, y_test_pred,
                                        target_names=["Ruim(0)", "Médio(1)", "Bom(2)"]))

    # ── Log no DagHub / MLflow ────────────────────────────────
    with mlflow.start_run(run_name=f"{STRATEGY}_{model_name}"):
        mlflow.log_param("strategy",      STRATEGY)
        mlflow.log_param("model",         model_name)
        mlflow.log_param("train_samples", int(len(X_train_smote)))
        mlflow.log_param("val_samples",   int(len(X_val)))
        mlflow.log_param("test_samples",  int(len(X_test)))
        mlflow.log_metrics(all_metrics)
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model",
            registered_model_name=f"wine_{STRATEGY}_{model_name}",
        )
    print(f"  ✅ Registrado → wine_{STRATEGY}_{model_name}")

print("\n🎉 SMOTE — todos os modelos registrados no DagHub!")



  Estratégia: SMOTE  |  Modelo: logistic_regression
  Validação → Acc:0.5531  P:0.5714  R:0.5531  F1:0.5438
  Teste     → Acc:0.5431  P:0.5593  R:0.5431  F1:0.5348

              precision    recall  f1-score   support

     Ruim(0)       0.63      0.68      0.65       477
    Médio(1)       0.56      0.36      0.44       567
      Bom(2)       0.42      0.70      0.53       256

    accuracy                           0.54      1300
   macro avg       0.54      0.58      0.54      1300
weighted avg       0.56      0.54      0.53      1300



2026/05/03 12:39:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:39:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'wine_SMOTE_logistic_regression'.
2026/05/03 12:39:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine_SMOTE_logistic_regression, version 1
Created version '1' of model 'wine_SMOTE_logistic_regression'.


🏃 View run SMOTE_logistic_regression at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/ca54fd436bfe497297ed109abf6295cd
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_SMOTE_logistic_regression

  Estratégia: SMOTE  |  Modelo: random_forest
  Validação → Acc:0.6877  P:0.6893  R:0.6877  F1:0.6866
  Teste     → Acc:0.6923  P:0.6945  R:0.6923  F1:0.6913

              precision    recall  f1-score   support

     Ruim(0)       0.75      0.78      0.76       477
    Médio(1)       0.69      0.61      0.65       567
      Bom(2)       0.61      0.72      0.66       256

    accuracy                           0.69      1300
   macro avg       0.68      0.70      0.69      1300
weighted avg       0.69      0.69      0.69      1300



2026/05/03 12:39:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:40:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'wine_SMOTE_random_forest'.
2026/05/03 12:40:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine_SMOTE_random_forest, version 1
Created version '1' of model 'wine_SMOTE_random_forest'.


🏃 View run SMOTE_random_forest at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/cceaa13afb2643b0bf178760df28e98e
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_SMOTE_random_forest

  Estratégia: SMOTE  |  Modelo: knn
  Validação → Acc:0.5885  P:0.5977  R:0.5885  F1:0.5852
  Teste     → Acc:0.5831  P:0.5897  R:0.5831  F1:0.5795

              precision    recall  f1-score   support

     Ruim(0)       0.64      0.66      0.65       477
    Médio(1)       0.59      0.46      0.52       567
      Bom(2)       0.50      0.70      0.58       256

    accuracy                           0.58      1300
   macro avg       0.58      0.61      0.58      1300
weighted avg       0.59      0.58      0.58      1300



...

🏃 View run SMOTE_svm at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/832fb91b9e3d4613994bc39820579569
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_SMOTE_svm

  Estratégia: SMOTE  |  Modelo: xgboost
  Validação → Acc:0.6746  P:0.6750  R:0.6746  F1:0.6741
  Teste     → Acc:0.6862  P:0.6877  R:0.6862  F1:0.6852

              precision    recall  f1-score   support

     Ruim(0)       0.73      0.75      0.74       477
    Médio(1)       0.68      0.61      0.64       567
      Bom(2)       0.62      0.73      0.67       256

    accuracy                           0.69      1300
   macro avg       0.68      0.70      0.69      1300
weighted avg       0.69      0.69      0.69      1300



2026/05/03 12:41:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:41:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'wine_SMOTE_xgboost'.
2026/05/03 12:42:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine_SMOTE_xgboost, version 1
Created version '1' of model 'wine_SMOTE_xgboost'.


🏃 View run SMOTE_xgboost at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/178f4691287c4600ad59c9f4f09658e9
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_SMOTE_xgboost

  Estratégia: SMOTE  |  Modelo: mlp
  Validação → Acc:0.6438  P:0.6483  R:0.6438  F1:0.6444
  Teste     → Acc:0.6369  P:0.6429  R:0.6369  F1:0.6376

              precision    recall  f1-score   support

     Ruim(0)       0.72      0.64      0.68       477
    Médio(1)       0.59      0.67      0.63       567
      Bom(2)       0.62      0.56      0.59       256

    accuracy                           0.64      1300
   macro avg       0.64      0.62      0.63      1300
weighted avg       0.64      0.64      0.64      1300



2026/05/03 12:43:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:43:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'wine_SMOTE_mlp'.
2026/05/03 12:43:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine_SMOTE_mlp, version 1
Created version '1' of model 'wine_SMOTE_mlp'.


🏃 View run SMOTE_mlp at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/20144f94dfa44dfbbac4f2514f888bc0
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_SMOTE_mlp

🎉 SMOTE — todos os modelos registrados no DagHub!


---
## Estratégia 2 — Tomek Links (Under-sampling)
**TomekLinks** requer features numéricas, portanto o pré-processamento  
é aplicado primeiro e compartilhado entre todos os modelos.


### 8. Pré-processar e aplicar Tomek Links

In [80]:
# ── Pré-processamento ────────────────────────────────────────
preprocessor_tomek = make_preprocessor()
X_train_trans = preprocessor_tomek.fit_transform(X_train)
X_val_trans   = preprocessor_tomek.transform(X_val)
X_test_trans  = preprocessor_tomek.transform(X_test)

y_train_int = y_train.values.astype(int)
y_val_int   = y_val.values.astype(int)
y_test_int  = y_test.values.astype(int)

# ── Tomek Links ───────────────────────────────────────────────
tomek = TomekLinks()
X_train_tomek, y_train_tomek = tomek.fit_resample(X_train_trans, y_train_int)

print("Distribuição após Tomek Links:")
print(pd.Series(y_train_tomek).value_counts().sort_index())
print(f"\nAmostras removidas: {len(X_train_trans) - len(X_train_tomek)}")


Distribuição após Tomek Links:
0    1233
1    1447
2     766
Name: count, dtype: int64

Amostras removidas: 451


### 9. Treinar, validar, testar e registrar — Tomek Links

In [81]:
STRATEGY = "Tomek"
results_tomek = {}

for model_name, model in get_models().items():
    print(f"\n{'='*60}")
    print(f"  Estratégia: {STRATEGY}  |  Modelo: {model_name}")
    print(f"{'='*60}")

    # ── Treino direto nos dados já transformados ───────────────
    model.fit(X_train_tomek, y_train_tomek)

    # ── Validação ─────────────────────────────────────────────
    y_val_pred  = model.predict(X_val_trans)
    val_metrics = compute_metrics(y_val_int, y_val_pred, prefix="val_")

    # ── Teste ─────────────────────────────────────────────────
    y_test_pred  = model.predict(X_test_trans)
    test_metrics = compute_metrics(y_test_int, y_test_pred, prefix="test_")

    all_metrics = {**val_metrics, **test_metrics}
    results_tomek[model_name] = all_metrics

    # ── Imprimir resultados ───────────────────────────────────
    print(f"  Validação → Acc:{val_metrics['val_accuracy']:.4f}  "
          f"P:{val_metrics['val_precision']:.4f}  "
          f"R:{val_metrics['val_recall']:.4f}  "
          f"F1:{val_metrics['val_f1']:.4f}")
    print(f"  Teste     → Acc:{test_metrics['test_accuracy']:.4f}  "
          f"P:{test_metrics['test_precision']:.4f}  "
          f"R:{test_metrics['test_recall']:.4f}  "
          f"F1:{test_metrics['test_f1']:.4f}")
    print("\n" + classification_report(y_test_int, y_test_pred,
                                        target_names=["Ruim(0)", "Médio(1)", "Bom(2)"]))

    # ── Log no DagHub / MLflow ────────────────────────────────
    with mlflow.start_run(run_name=f"{STRATEGY}_{model_name}"):
        mlflow.log_param("strategy",      STRATEGY)
        mlflow.log_param("model",         model_name)
        mlflow.log_param("train_samples", int(len(X_train_tomek)))
        mlflow.log_param("val_samples",   int(len(X_val_trans)))
        mlflow.log_param("test_samples",  int(len(X_test_trans)))
        mlflow.log_metrics(all_metrics)

        # Modelo treinado
        mlflow.sklearn.log_model(
            model,
            artifact_path="model",
            registered_model_name=f"wine_{STRATEGY}_{model_name}",
        )
        # Pré-processador compartilhado (necessário para inferência futura)
        mlflow.sklearn.log_model(
            preprocessor_tomek,
            artifact_path="preprocessor",
        )
    print(f"  ✅ Registrado → wine_{STRATEGY}_{model_name}")

print("\n🎉 Tomek — todos os modelos registrados no DagHub!")



  Estratégia: Tomek  |  Modelo: logistic_regression
  Validação → Acc:0.5846  P:0.5839  R:0.5846  F1:0.5821
  Teste     → Acc:0.5792  P:0.5830  R:0.5792  F1:0.5775

              precision    recall  f1-score   support

     Ruim(0)       0.65      0.62      0.63       477
    Médio(1)       0.53      0.61      0.57       567
      Bom(2)       0.57      0.42      0.49       256

    accuracy                           0.58      1300
   macro avg       0.58      0.55      0.56      1300
weighted avg       0.58      0.58      0.58      1300



2026/05/03 12:43:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:43:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'wine_Tomek_logistic_regression'.
2026/05/03 12:43:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine_Tomek_logistic_regression, version 1
Created version '1' of model 'wine_Tomek_logistic_regression'.
2026/05/03 12:43:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:44:09 WARNING mlflow.sklearn: Saving scikit-learn models 

🏃 View run Tomek_logistic_regression at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/cceff10e6a874eb9b4885312c9445999
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_Tomek_logistic_regression

  Estratégia: Tomek  |  Modelo: random_forest
  Validação → Acc:0.6746  P:0.6754  R:0.6746  F1:0.6749
  Teste     → Acc:0.6900  P:0.6915  R:0.6900  F1:0.6906

              precision    recall  f1-score   support

     Ruim(0)       0.76      0.73      0.74       477
    Médio(1)       0.66      0.67      0.66       567
      Bom(2)       0.65      0.67      0.66       256

    accuracy                           0.69      1300
   macro avg       0.69      0.69      0.69      1300
weighted avg       0.69      0.69      0.69      1300



2026/05/03 12:44:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:44:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'wine_Tomek_random_forest'.
2026/05/03 12:44:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine_Tomek_random_forest, version 1
Created version '1' of model 'wine_Tomek_random_forest'.
2026/05/03 12:44:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:45:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or c

🏃 View run Tomek_random_forest at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/21817dc5ce3942639e3313c66fc9b356
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_Tomek_random_forest

  Estratégia: Tomek  |  Modelo: knn
  Validação → Acc:0.6131  P:0.6118  R:0.6131  F1:0.6115
  Teste     → Acc:0.6000  P:0.6003  R:0.6000  F1:0.6001

              precision    recall  f1-score   support

     Ruim(0)       0.65      0.64      0.64       477
    Médio(1)       0.58      0.58      0.58       567
      Bom(2)       0.57      0.57      0.57       256

    accuracy                           0.60      1300
   macro avg       0.60      0.60      0.60      1300
weighted avg       0.60      0.60      0.60      1300



...

🏃 View run Tomek_svm at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/758efaecadfb47fdbdf7939af2902d9a
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_Tomek_svm

  Estratégia: Tomek  |  Modelo: xgboost
  Validação → Acc:0.6723  P:0.6722  R:0.6723  F1:0.6723
  Teste     → Acc:0.6862  P:0.6867  R:0.6862  F1:0.6862

              precision    recall  f1-score   support

     Ruim(0)       0.74      0.73      0.74       477
    Médio(1)       0.67      0.65      0.66       567
      Bom(2)       0.64      0.68      0.66       256

    accuracy                           0.69      1300
   macro avg       0.68      0.69      0.68      1300
weighted avg       0.69      0.69      0.69      1300



2026/05/03 12:47:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:47:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'wine_Tomek_xgboost'.
2026/05/03 12:47:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine_Tomek_xgboost, version 1
Created version '1' of model 'wine_Tomek_xgboost'.
2026/05/03 12:47:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:47:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format 

🏃 View run Tomek_xgboost at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/c23886f8b9814ba9b8d1b5b850ae5f1b
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_Tomek_xgboost

  Estratégia: Tomek  |  Modelo: mlp
  Validação → Acc:0.6538  P:0.6594  R:0.6538  F1:0.6529
  Teste     → Acc:0.6423  P:0.6478  R:0.6423  F1:0.6419

              precision    recall  f1-score   support

     Ruim(0)       0.71      0.66      0.68       477
    Médio(1)       0.63      0.58      0.60       567
      Bom(2)       0.57      0.75      0.65       256

    accuracy                           0.64      1300
   macro avg       0.64      0.66      0.65      1300
weighted avg       0.65      0.64      0.64      1300



2026/05/03 12:48:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:48:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'wine_Tomek_mlp'.
2026/05/03 12:48:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: wine_Tomek_mlp, version 1
Created version '1' of model 'wine_Tomek_mlp'.
2026/05/03 12:48:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:49:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exe

🏃 View run Tomek_mlp at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1/runs/327f1bfb198a4388bc4b9dbbb4cc053d
🧪 View experiment at: https://dagshub.com/frpbotero/wine-quality.mlflow/#/experiments/1
  ✅ Registrado → wine_Tomek_mlp

🎉 Tomek — todos os modelos registrados no DagHub!


---
## 10. Resumo Final

In [82]:
import pandas as pd

rows = []
for model_name in get_models().keys():
    sm = results_smote.get(model_name, {})
    tk = results_tomek.get(model_name, {})
    rows.append({
        "Modelo":          model_name,
        "SMOTE  Val-F1":   sm.get("val_f1",  "-"),
        "SMOTE  Test-F1":  sm.get("test_f1", "-"),
        "SMOTE  Test-Acc": sm.get("test_accuracy", "-"),
        "Tomek  Val-F1":   tk.get("val_f1",  "-"),
        "Tomek  Test-F1":  tk.get("test_f1", "-"),
        "Tomek  Test-Acc": tk.get("test_accuracy", "-"),
    })

summary = pd.DataFrame(rows).set_index("Modelo")
print(summary.to_string())
print("\n🏆 Melhor Test-F1 SMOTE :", summary["SMOTE  Test-F1"].max(),
      "→", summary["SMOTE  Test-F1"].idxmax())
print("🏆 Melhor Test-F1 Tomek :", summary["Tomek  Test-F1"].max(),
      "→", summary["Tomek  Test-F1"].idxmax())


                     SMOTE  Val-F1  SMOTE  Test-F1  SMOTE  Test-Acc  Tomek  Val-F1  Tomek  Test-F1  Tomek  Test-Acc
Modelo                                                                                                             
logistic_regression         0.5438          0.5348           0.5431         0.5821          0.5775           0.5792
random_forest               0.6866          0.6913           0.6923         0.6749          0.6906           0.6900
knn                         0.5852          0.5795           0.5831         0.6115          0.6001           0.6000
svm                         0.5820          0.5961           0.6008         0.6011          0.6108           0.6123
xgboost                     0.6741          0.6852           0.6862         0.6723          0.6862           0.6862
mlp                         0.6444          0.6376           0.6369         0.6529          0.6419           0.6423

🏆 Melhor Test-F1 SMOTE : 0.6913 → random_forest
🏆 Melhor Test-F1 Tomek 

In [83]:

# # salvar pipeline treinado
# with open("mlp_wine.pkl", "wb") as f:
#     pickle.dump(mlp_pipeline, f)
# print("Modelo salvo em mlp_wine.pkl")


In [84]:
# # treinar classificador diretamente sobre os dados já transformados
# from sklearn.neural_network import MLPClassifier
# clf = MLPClassifier(hidden_layer_sizes=(200,100), max_iter=2000, random_state=42)
# clf.fit(X_train_tomek, y_train_tomek)

# # avaliar usando X_test_trans
# y_pred = clf.predict(X_test_trans)
# print("Accuracy:", accuracy_score(y_test_num, y_pred))
# print(classification_report(y_test_num, y_pred))